In [1]:
import os

REQ_PATH = "requirements.txt" if os.path.exists("requirements.txt") else "../requirements.txt"
!pip install -qU -r {REQ_PATH}


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import uuid

from typing import TypedDict, Annotated

from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from qdrant_client.models import Distance, VectorParams, PointStruct
from langchain_core.output_parsers import StrOutputParser

import fitz

c:\Users\lenovo\OneDrive\Desktop\Projects\agentic-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DOCS_DIR = "docs"
MARKDOWN_DIR = "markdown_docs"
PARENT_STORE_PATH = "parent_store"
QDRANT_PATH = "qdrant_db"

CHILD_COLLECTION = "document_child_chunks"

RETRIEVAL_SCORE_THRESHOLD = 0.4
DEFAULT_RETRIEVAL_K = 7

os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(MARKDOWN_DIR, exist_ok=True)
os.makedirs(PARENT_STORE_PATH, exist_ok=True)
os.makedirs(QDRANT_PATH, exist_ok=True)

In [4]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

In [5]:
print(llm.invoke("What is 2+2?").content)

2 + 2 = 4.


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3570.81it/s]


In [7]:
# Initialize the client ONLY if it doesn't already exist
if 'client' not in globals():
    client = QdrantClient(path=QDRANT_PATH)

from qdrant_client.models import Distance, VectorParams

collections = [c.name for c in client.get_collections().collections]

if CHILD_COLLECTION not in collections:
    client.create_collection(
        collection_name=CHILD_COLLECTION,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

In [8]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=CHILD_COLLECTION,
    embedding=embeddings  
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [9]:
def pdf_to_text(pdf_path):
    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
pdf_path = "docs/sample.pdf"

text = pdf_to_text(pdf_path)

chunks = splitter.split_text(text)

print(len(chunks))

38


In [11]:
points = []

for chunk in chunks:

    vector = embeddings.embed_query(chunk)

    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vector,
            payload={
                "text": chunk
            }
        )
    )
    
client.upsert(
    collection_name=CHILD_COLLECTION,
    points=points
)


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [12]:
@tool
def search_child_chunks(query: str, limit: int = 3):
    """
    Search indexed document chunks.
    """

    query_vector = embeddings.embed_query(query)

    results = client.query_points(
        collection_name=CHILD_COLLECTION,
        query=query_vector,
        limit=limit
    )

    output = []

    for r in results.points:
        output.append(r.payload["text"])

    return "\n\n".join(output)

In [13]:
print(
    search_child_chunks.invoke(
        {
            "query": "What is this document about?"
        }
    )
)

indication of this would be a solid 5 on the BC calculus, an A* on British A-levels, or some other indicator of 
full mastery of single-variable calculus. A lot depends on how confident you are in your mastery of single 
variable calculus at the level of the end of Math 104. If you have studied Taylor series, then you are 
probably at or close to this level. The most common place is to start in Math 201 and then take Math 202 in

indication of this would be a solid 5 on the BC calculus, an A* on British A-levels, or some other indicator of 
full mastery of single-variable calculus. A lot depends on how confident you are in your mastery of single 
variable calculus at the level of the end of Math 104. If you have studied Taylor series, then you are 
probably at or close to this level. The most common place is to start in Math 201 and then take Math 202 in

indication of this would be a solid 5 on the BC calculus, an A* on British A-levels, or some other indicator of 
full mastery of sin

In [14]:
class AgentState(TypedDict):
    question: str
    context: str
    answer: str

In [15]:
def retrieve(state):

    context = search_child_chunks.invoke(
        {
            "query": state["question"]
        }
    )

    return {
        "context": context
    }

In [16]:
def generate(state):

    prompt = f"""
Answer ONLY using the provided context.

Context:
{state['context']}

Question:
{state['question']}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [17]:
import gradio as gr
from typing import TypedDict
from langgraph.graph import StateGraph, END


class GraphState(TypedDict):
    question: str
    context: str
    answer: str



def retrieve_node(state: GraphState):
    """Fetches the child chunks from Qdrant based on the user's question."""
    print("---RETRIEVING CONTEXT---")
    
    retrieved_context = search_child_chunks.invoke(state["question"])
    
    return {"context": retrieved_context}

def generate_node(state: GraphState):
    """Passes the context and question to Ollama to generate an answer."""
    print("---GENERATING ANSWER---")
    
    prompt_text = f"""
Answer ONLY using the provided context.

Context:
{state['context']}

Question:
{state['question']}
"""
    
    response = llm.invoke(prompt_text)
    
    return {"answer": response.content}

# --- 3. Build and Compile the Agentic Graph ---
workflow = StateGraph(GraphState)

# Add our nodes to the graph
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)

# Define the flow (The logic path the agent takes)
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile the graph into an executable application
agent_app = workflow.compile()

# --- 4. The Interface ---
def chat_fn(message, history):
    result = agent_app.invoke({"question": message})
    return result["answer"]

choco_theme = gr.themes.Soft(
    primary_hue="amber",
    neutral_hue="stone"
).set(
    body_background_fill="#4A3B32", 
    block_background_fill="#5C4A3D",
    body_text_color="#F5E6D3",
    button_primary_background_fill="#8B5A2B",
    button_primary_text_color="#FFFFFF"
)

with gr.Blocks(theme=choco_theme) as demo:
    gr.ChatInterface(
        fn=chat_fn, 
        title="Agentic Document Explorer",
        description="A LangGraph-powered retrieval assistant utilizing hierarchical Qdrant indexing."
    )

if __name__ == "__main__":
    demo.launch()


C:\Users\lenovo\AppData\Local\Temp\ipykernel_1620\1085190070.py:70: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=choco_theme) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\lenovo\OneDrive\Desktop\Projects\agentic-rag\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


---RETRIEVING CONTEXT---
---GENERATING ANSWER---


c:\Users\lenovo\OneDrive\Desktop\Projects\agentic-rag\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
